In [ ]:
pip install llama-cpp-python==0.2.90 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121

In [ ]:
# ============================================================
#  TEXT-to-SQL ZERO-SHOT – LOCAL  (v5 – tối ưu tốc độ)
#  Fix v5:
#     - Pre-cache toàn bộ gold SQL results trước khi inference
#       → không phải exec gold SQL lặp lại trong loop
#     - Tách EX calculation ra sau inference (không block GPU)
#     - Giữ nguyên CR_P + FK + Rule + difficulty từ sql struct
#  v5 + VRAM:
#     - Đổi model → Llama-2-7B-Chat Q4_0
#     - Đo VRAM peak + VRAM trung bình qua pynvml
# ============================================================

import os, json, re, time, zipfile, shutil, urllib.request
import sqlite3, subprocess
import nltk
from tqdm.auto import tqdm
from llama_cpp import Llama

import pynvml
pynvml.nvmlInit()
_nvml_handle  = pynvml.nvmlDeviceGetHandleByIndex(0)
_vram_samples = []

# ── 1. CONFIG ────────────────────────────────────────────────
MODEL_PATH = r"D:\Lap_46T\Llama-2-7b\llama-2-7b-chat.Q4_0.gguf"
SPIDER_DIR = r"D:\Lap_46T\T5-small\spider_data"

# Đổi tên thư mục lưu kết quả để phân biệt với Mistral
RESULT_DIR = r"D:\Lap_46T\Llama-2-7b\llama-2-7b-few-shot_k3" 

os.makedirs(RESULT_DIR, exist_ok=True)
EVAL_SCRIPT = os.path.join(RESULT_DIR, "evaluation.py")
PROC_SCRIPT = os.path.join(RESULT_DIR, "process_sql.py")

CFG = dict(
    n_ctx          = 4096,   # Llama-2 hỗ trợ tối đa 4096 tokens
    n_gpu_layers   = 35,     # 35 layers là đủ offload toàn bộ model 7B lên GPU
    max_tokens     = 256,    # Độ dài tối đa cho câu SQL output
    temperature    = 0.0,    # Cần giữ 0.0 (Greedy Decoding) cho bài toán code/SQL
    top_p          = 1.0,
    repeat_penalty = 1.1,
)

# ── 2. DOWNLOAD SPIDER (nếu chưa có) ─────────────────────────
def download_spider():
    os.environ['KAGGLE_USERNAME'] = "phankhaclap"
    os.environ['KAGGLE_KEY']      = "0ba946628cb1f5acb76ecd357f590e95"
    subprocess.run([
        "kaggle", "datasets", "download",
        "-d", "jeromeblanchet/yale-universitys-spider-10-nlp-dataset",
        "-p", RESULT_DIR
    ], check=True)
    zip_path = os.path.join(RESULT_DIR,
                "yale-universitys-spider-10-nlp-dataset.zip")
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(os.path.join(RESULT_DIR, "temp"))
    src = os.path.join(RESULT_DIR, "temp", "spider")
    if not os.path.exists(src):
        src = os.path.join(RESULT_DIR, "temp")
    if os.path.exists(SPIDER_DIR):
        shutil.rmtree(SPIDER_DIR)
    shutil.move(src, SPIDER_DIR)
    shutil.rmtree(os.path.join(RESULT_DIR, "temp"), ignore_errors=True)
    os.remove(zip_path)

if not os.path.exists(SPIDER_DIR):
    download_spider()

# ── 3. DOWNLOAD OFFICIAL EVAL ────────────────────────────────
for fpath, url in [
    (EVAL_SCRIPT,
     "https://raw.githubusercontent.com/taoyds/spider/master/evaluation.py"),
    (PROC_SCRIPT,
     "https://raw.githubusercontent.com/taoyds/spider/master/process_sql.py"),
]:
    if not os.path.exists(fpath):
        urllib.request.urlretrieve(url, fpath)

nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)
print(f"✅ Spider data : {SPIDER_DIR}  model_1_zeroshot_llama.ipynb:79  model_1_zeroshot_llama27b.ipynb:79  model_2_zeroshot_mistral7b.ipynb:79  model_2_fewshot_mistral.ipynb:79  model_3_fewshot_mistral.ipynb:79  model_4_fewshot_mistral_k1.ipynb:79  model_5_fewshot_mistral_k2.ipynb:79  model_fewshot_llama2chat7b_k3.ipynb:81  model_fewshot_llama2chat7b_k1.ipynb:81 - model_few-shot_llama-2-chat-7b_k5.ipynb:81")

# ── 4. HELPERS ───────────────────────────────────────────────
def load_json(path):
    with open(path, encoding='utf-8') as f:
        return json.load(f)

def normalize_sql(sql: str) -> str:
    sql = sql.lower().strip()
    sql = re.sub(r'\s+', ' ', sql)
    sql = re.sub(r'\s*([,\(\)])\s*', r' \1 ', sql)
    return re.sub(r'\s+', ' ', sql).strip()

# ── 5. DIFFICULTY từ SQL struct ───────────────────────────────
def count_components(s: dict) -> dict:
    def _nested(s):
        if not isinstance(s, dict):
            return 0
        count = 0
        for tbl in s.get("from", {}).get("table_units", []):
            if isinstance(tbl, list) and len(tbl) == 2 and tbl[0] == "sql":
                count += 1 + _nested(tbl[1])
        for clause in ["where", "having"]:
            for cond in s.get(clause, []):
                if isinstance(cond, list) and len(cond) >= 3:
                    for v in cond[2:4]:
                        if isinstance(v, dict) and "from" in v:
                            count += 1 + _nested(v)
        for key in ["intersect", "union", "except"]:
            sub = s.get(key)
            if sub:
                count += 1 + _nested(sub)
        return count

    nested    = _nested(s)
    where_n   = len([c for c in s.get("where", []) if not isinstance(c, str)])
    has_group = bool(s.get("groupBy"))
    has_order = bool(s.get("orderBy"))
    has_have  = bool(s.get("having"))
    has_limit = s.get("limit") is not None
    has_setop = any(s.get(k) for k in ["intersect","union","except"])

    return dict(nested=nested, where_n=where_n, groupby=has_group,
                orderby=has_order, having=has_have, limit=has_limit,
                setop=has_setop)

def classify_difficulty(sql_struct: dict) -> str:
    c = count_components(sql_struct)
    if c["nested"] >= 2:                                      return "extra"
    if c["nested"] == 1:                                      return "hard"
    if c["where_n"] >= 4:                                     return "hard"
    if c["having"] and c["groupby"] and c["where_n"] >= 2:   return "hard"
    if c["setop"] or c["groupby"] or c["having"]:             return "medium"
    if c["where_n"] >= 2:                                     return "medium"
    if c["orderby"] and c["where_n"] >= 1:                    return "medium"
    if c["limit"]:                                            return "medium"
    return "easy"

def get_difficulty(item: dict) -> str:
    d = item.get("difficulty", "")
    if d and d not in ("MISSING", ""):
        return d.lower().strip()
    if "sql" in item:
        return classify_difficulty(item["sql"])
    return "unknown"

# ── 6. LOAD DATA ─────────────────────────────────────────────
tables_path = os.path.join(SPIDER_DIR, "tables.json")
dev_path    = os.path.join(SPIDER_DIR, "dev.json")
dev_data    = load_json(dev_path)
# ── FEW-SHOT EXAMPLES (k=3) ───────────────────────────────
train_path = os.path.join(SPIDER_DIR, "train_spider.json")
train_data = load_json(train_path)

few_shot_examples = train_data[:5]
schema_map  = {}

# Build schema CR_P
for db in load_json(tables_path):
    db_id, tables = db['db_id'], db['table_names_original']
    cols, col_types = db['column_names_original'], db['column_types']
    pks = set(db['primary_keys'])
    fk_lookup = {}
    for c1, c2 in db['foreign_keys']:
        fk_lookup[c1] = (tables[cols[c2][0]], cols[c2][1])
    parts = []
    for t_idx, t_name in enumerate(tables):
        col_defs, fk_defs = [], []
        for c_idx, (t_id, c_name) in enumerate(cols):
            if t_id != t_idx or c_name == '*':
                continue
            line = f"  {c_name} {col_types[c_idx]}"
            if c_idx in pks:
                line += " primary key"
            col_defs.append(line)
            if c_idx in fk_lookup:
                ref_t, ref_c = fk_lookup[c_idx]
                fk_defs.append(
                    f"  foreign key ({c_name}) references {ref_t}({ref_c})")
        parts.append(
            f"CREATE TABLE {t_name} (\n"
            + ",\n".join(col_defs + fk_defs) + "\n);")
    schema_map[db_id] = "\n\n".join(parts)

print(f"Dev samples  : {len(dev_data)}  model_1_zeroshot_llama.ipynb:178  model_1_zeroshot_llama27b.ipynb:178  model_2_zeroshot_mistral7b.ipynb:178  model_2_fewshot_mistral.ipynb:183  model_3_fewshot_mistral.ipynb:183  model_4_fewshot_mistral_k1.ipynb:183  model_5_fewshot_mistral_k2.ipynb:183  model_fewshot_llama2chat7b_k3.ipynb:185  model_fewshot_llama2chat7b_k1.ipynb:185 - model_few-shot_llama-2-chat-7b_k5.ipynb:185")

from collections import Counter
dist = Counter(get_difficulty(it) for it in dev_data)
print(f"Difficulty   : {dict(sorted(dist.items()))}  model_1_zeroshot_llama.ipynb:182  model_1_zeroshot_llama27b.ipynb:182  model_2_zeroshot_mistral7b.ipynb:182  model_2_fewshot_mistral.ipynb:187  model_3_fewshot_mistral.ipynb:187  model_4_fewshot_mistral_k1.ipynb:187  model_5_fewshot_mistral_k2.ipynb:187  model_fewshot_llama2chat7b_k3.ipynb:189  model_fewshot_llama2chat7b_k1.ipynb:189 - model_few-shot_llama-2-chat-7b_k5.ipynb:189")

# ── 7. PRE-CACHE GOLD SQL RESULTS ────────────────────────────
DB_DIR = os.path.join(SPIDER_DIR, "database")

def exec_sql(sql: str, db_id: str):
    db_path = os.path.join(DB_DIR, db_id, f"{db_id}.sqlite")
    try:
        conn = sqlite3.connect(db_path)
        conn.text_factory = lambda b: b.decode(errors="ignore")
        cur = conn.cursor()
        cur.execute(sql)
        rows = cur.fetchall()
        conn.close()
        return frozenset(rows)
    except Exception:
        return None

print(">>> Precaching gold SQL results …  model_1_zeroshot_llama.ipynb:200  model_1_zeroshot_llama27b.ipynb:200  model_2_zeroshot_mistral7b.ipynb:200  model_2_fewshot_mistral.ipynb:205  model_3_fewshot_mistral.ipynb:205  model_4_fewshot_mistral_k1.ipynb:205  model_5_fewshot_mistral_k2.ipynb:205  model_fewshot_llama2chat7b_k3.ipynb:207  model_fewshot_llama2chat7b_k1.ipynb:207 - model_few-shot_llama-2-chat-7b_k5.ipynb:207")
gold_cache = {}   # key: (gold_sql, db_id) → frozenset | None
for item in tqdm(dev_data, desc="Cache gold", ncols=80, leave=False):
    key = (item['query'], item['db_id'])
    if key not in gold_cache:
        gold_cache[key] = exec_sql(item['query'], item['db_id'])
print(f"✅ Cached {len(gold_cache)} unique gold queries\n  model_1_zeroshot_llama.ipynb:206  model_1_zeroshot_llama27b.ipynb:206  model_2_zeroshot_mistral7b.ipynb:206  model_2_fewshot_mistral.ipynb:211  model_3_fewshot_mistral.ipynb:211  model_4_fewshot_mistral_k1.ipynb:211  model_5_fewshot_mistral_k2.ipynb:211  model_fewshot_llama2chat7b_k3.ipynb:213  model_fewshot_llama2chat7b_k1.ipynb:213 - model_few-shot_llama-2-chat-7b_k5.ipynb:213")


def format_example(ex):
    return (
        f"Q: {ex['question']}\n"
        f"A: ```sql\n{ex['query']}\n```\n" # Ép bọc trong markdown
    )
# ── 8. PROMPT (Sửa cấu trúc theo template chuẩn của Llama-2-Chat) ──
def build_prompt(question: str, schema: str) -> str:

    examples_text = "\n\n".join(
        format_example(ex)
        for ex in few_shot_examples
    )

    return (
        f"[INST] "
        f"You are an expert Text-to-SQL model.\n"
        f"Generate only valid SQLite SQL query.\n"
        f"No explanation.\n\n"

        f"Here are some examples:\n\n"
        f"{examples_text}\n\n"

        f"Now solve this.\n\n"

        f"Database schema:\n"
        f"{schema}\n\n"

        f"Question:\n"
        f"{question}\n"
        f"[/INST]\n"
        f"```sql\n"
    )

# ── 9. EXTRACT SQL ───────────────────────────────────────────
def extract_sql(raw: str) -> str:
    # 1. Thử tìm SQL trong block markdown
    m = re.search(r'```sql\s*(.*?)\s*```', raw, re.DOTALL | re.IGNORECASE)
    if not m:
        # 2. Thử tìm block markdown chung chung
        m = re.search(r'```\s*(.*?)\s*```', raw, re.DOTALL)
    
    if m:
        raw = m.group(1)
    
    # 3. Loại bỏ các từ khóa giải thích thường gặp ở đầu/cuối
    raw = re.sub(r'^(Here is the SQL query.*?:\s*|Sure.*?:\s*|The SQL query.*?:\s*)', '', raw, flags=re.IGNORECASE | re.DOTALL)
    
    # Dọn dẹp khoảng trắng và newline
    raw = re.sub(r'\n+', ' ', raw).strip()
    
    # 4. Đảm bảo bắt đầu bằng SELECT
    if not re.match(r'^\s*SELECT', raw, re.IGNORECASE):
        # Nếu mô hình trả về "SELECT ...", ta giữ nguyên, nếu không thì ép thêm.
        # (Lưu ý: Đôi khi Llama-2 bị lỗi bỏ chữ S ở đầu, thành "ELECT...")
        if re.match(r'^\s*ELECT', raw, re.IGNORECASE):
            raw = "S" + raw
        else:
            raw = "SELECT " + raw

    # 5. Cắt bỏ mọi thứ sau dấu chấm phẩy
    if ';' in raw:
        raw = raw[:raw.index(';')]

    raw = raw.strip()
    return raw if raw and raw.upper() != "SELECT" else ""

# ── 10. LOAD MODEL ───────────────────────────────────────────
print(f">>> Loading model : {MODEL_PATH}  model_1_zeroshot_llama.ipynb:243  model_1_zeroshot_llama27b.ipynb:241  model_2_zeroshot_mistral7b.ipynb:245  model_2_fewshot_mistral.ipynb:277  model_3_fewshot_mistral.ipynb:273  model_4_fewshot_mistral_k1.ipynb:271  model_5_fewshot_mistral_k2.ipynb:271  model_fewshot_llama2chat7b_k3.ipynb:283  model_fewshot_llama2chat7b_k1.ipynb:283 - model_few-shot_llama-2-chat-7b_k5.ipynb:283")
assert os.path.exists(MODEL_PATH), f"❌ Not found: {MODEL_PATH}"
llm = Llama(
    model_path   = MODEL_PATH,
    n_ctx        = CFG['n_ctx'],
    n_gpu_layers = CFG['n_gpu_layers'],
    verbose      = False,
)
print("✅ Model loaded.\n  model_1_zeroshot_llama.ipynb:251  model_1_zeroshot_llama27b.ipynb:249  model_2_zeroshot_mistral7b.ipynb:253  model_2_fewshot_mistral.ipynb:285  model_3_fewshot_mistral.ipynb:281  model_4_fewshot_mistral_k1.ipynb:279  model_5_fewshot_mistral_k2.ipynb:279  model_fewshot_llama2chat7b_k3.ipynb:291  model_fewshot_llama2chat7b_k1.ipynb:291 - model_few-shot_llama-2-chat-7b_k5.ipynb:291")

# ── 11. INFERENCE LOOP ───────────────────────────────────────
DIFF_KEYS = ["easy", "medium", "hard", "extra"]
stats = {d: {"em": 0, "ex": 0, "n": 0} for d in DIFF_KEYS + ["unknown"]}

predictions, gold_lines = [], []
latencies   = []
empty_count = 0
debug_log   = []

pred_cache = []   # list of (pred_sql, gold_sql, db_id, diff_key, em_hit)

print(f">>> Zeroshot inference ({len(dev_data)} samples) …  model_1_zeroshot_llama.ipynb:264  model_1_zeroshot_llama27b.ipynb:262  model_2_zeroshot_mistral7b.ipynb:266  model_2_fewshot_mistral.ipynb:298  model_3_fewshot_mistral.ipynb:294  model_4_fewshot_mistral_k1.ipynb:292  model_5_fewshot_mistral_k2.ipynb:292  model_fewshot_llama2chat7b_k3.ipynb:304  model_fewshot_llama2chat7b_k1.ipynb:304 - model_few-shot_llama-2-chat-7b_k5.ipynb:304")
t_infer_start = time.perf_counter()

for i, item in enumerate(tqdm(dev_data, desc="Inference", ncols=80)):
    db_id    = item['db_id']
    question = item['question']
    gold_sql = item['query']
    schema   = schema_map.get(db_id, "")
    prompt   = build_prompt(question, schema)
    diff     = get_difficulty(item)
    diff_key = diff if diff in DIFF_KEYS else "unknown"

    t0 = time.perf_counter()
    out = llm(
        prompt,
        max_tokens     = CFG['max_tokens'],
        temperature    = CFG['temperature'],
        top_p          = CFG['top_p'],
        repeat_penalty = CFG['repeat_penalty'],
        stop = ["</s>", "[INST]", "[/INST]"]
    )
    t1 = time.perf_counter()
    latencies.append(t1 - t0)
    _vram_samples.append(
        pynvml.nvmlDeviceGetMemoryInfo(_nvml_handle).used / (1024 ** 2)
    )

    raw_text = out['choices'][0]['text']
    pred_sql = extract_sql(raw_text)
    if not pred_sql:
        empty_count += 1
        pred_sql = "SELECT 1"

    em_hit = int(normalize_sql(pred_sql) == normalize_sql(gold_sql))

    if i < 5:
        debug_log.append({
            "idx": i, "question": question, "difficulty": diff,
            "gold": gold_sql, "pred": pred_sql,
            "raw": raw_text, "em": em_hit,
        })

    pred_cache.append((pred_sql, gold_sql, db_id, diff_key, em_hit))
    predictions.append(pred_sql + "\n")
    gold_lines.append(f"{gold_sql}\t{db_id}\n")

t_infer_end = time.perf_counter()
infer_time  = t_infer_end - t_infer_start
print(f"✅ Inference done in {infer_time/60:.1f} min\n  model_1_zeroshot_llama.ipynb:312  model_1_zeroshot_llama27b.ipynb:310  model_2_zeroshot_mistral7b.ipynb:314  model_2_fewshot_mistral.ipynb:346  model_3_fewshot_mistral.ipynb:342  model_4_fewshot_mistral_k1.ipynb:340  model_5_fewshot_mistral_k2.ipynb:340  model_fewshot_llama2chat7b_k3.ipynb:352  model_fewshot_llama2chat7b_k1.ipynb:352 - model_few-shot_llama-2-chat-7b_k5.ipynb:352")

vram_peak_mb = max(_vram_samples)
vram_avg_mb  = sum(_vram_samples) / len(_vram_samples)

# ── 12. TÍNH EX SAU INFERENCE ──────────────────────────────
print(">>> Computing Execution Accuracy …  model_1_zeroshot_llama.ipynb:318  model_1_zeroshot_llama27b.ipynb:316  model_2_zeroshot_mistral7b.ipynb:320  model_2_fewshot_mistral.ipynb:352  model_3_fewshot_mistral.ipynb:348  model_4_fewshot_mistral_k1.ipynb:346  model_5_fewshot_mistral_k2.ipynb:346  model_fewshot_llama2chat7b_k3.ipynb:358  model_fewshot_llama2chat7b_k1.ipynb:358 - model_few-shot_llama-2-chat-7b_k5.ipynb:358")
for pred_sql, gold_sql, db_id, diff_key, em_hit in tqdm(
        pred_cache, desc="Exec eval", ncols=80, leave=False):
    pred_res = exec_sql(pred_sql, db_id)
    gold_res = gold_cache.get((gold_sql, db_id))
    ex_hit   = int(
        pred_res is not None and gold_res is not None
        and pred_res == gold_res
    )
    stats[diff_key]["em"] += em_hit
    stats[diff_key]["ex"] += ex_hit
    stats[diff_key]["n"]  += 1
print("✅ EX computed.\n  model_1_zeroshot_llama.ipynb:330  model_1_zeroshot_llama27b.ipynb:328  model_2_zeroshot_mistral7b.ipynb:332  model_2_fewshot_mistral.ipynb:364  model_3_fewshot_mistral.ipynb:360  model_4_fewshot_mistral_k1.ipynb:358  model_5_fewshot_mistral_k2.ipynb:358  model_fewshot_llama2chat7b_k3.ipynb:370  model_fewshot_llama2chat7b_k1.ipynb:370 - model_few-shot_llama-2-chat-7b_k5.ipynb:370")

# ── 13. SAVE PRED / GOLD ─────────────────────────────────────
pred_path = os.path.join(RESULT_DIR, "pred_zero_shot_crp.txt")
gold_path = os.path.join(RESULT_DIR, "gold_zero_shot_crp.txt")
with open(pred_path, 'w', encoding='utf-8') as f: f.writelines(predictions)
with open(gold_path, 'w', encoding='utf-8') as f: f.writelines(gold_lines)
with open(os.path.join(RESULT_DIR, "debug_samples.json"),
          'w', encoding='utf-8') as f:
    json.dump(debug_log, f, indent=2, ensure_ascii=False)

# ── 14. PATCH + RUN OFFICIAL EVAL ────────────────────────────
with open(EVAL_SCRIPT, "r", encoding="utf-8") as f:
    ec = f.read()
if 'text_factory' not in ec:
    ec = ec.replace(
        'conn = sqlite3.connect(db)',
        'conn = sqlite3.connect(db)\n'
        '    conn.text_factory = lambda b: b.decode(errors="ignore")'
    )
    with open(EVAL_SCRIPT, "w", encoding="utf-8") as f:
        f.write(ec)

print(">>> Spider Official Evaluation:  model_1_zeroshot_llama.ipynb:353  model_1_zeroshot_llama27b.ipynb:351  model_2_zeroshot_mistral7b.ipynb:355  model_2_fewshot_mistral.ipynb:387  model_3_fewshot_mistral.ipynb:383  model_4_fewshot_mistral_k1.ipynb:381  model_5_fewshot_mistral_k2.ipynb:381  model_fewshot_llama2chat7b_k3.ipynb:393  model_fewshot_llama2chat7b_k1.ipynb:393 - model_few-shot_llama-2-chat-7b_k5.ipynb:393")
print("─  model_1_zeroshot_llama.ipynb:354  model_1_zeroshot_llama27b.ipynb:352  model_2_zeroshot_mistral7b.ipynb:356  model_2_fewshot_mistral.ipynb:388  model_3_fewshot_mistral.ipynb:384  model_4_fewshot_mistral_k1.ipynb:382  model_5_fewshot_mistral_k2.ipynb:382  model_fewshot_llama2chat7b_k3.ipynb:394  model_fewshot_llama2chat7b_k1.ipynb:394 - model_few-shot_llama-2-chat-7b_k5.ipynb:394" * 62)
os.system(
    f'python "{EVAL_SCRIPT}" '
    f'--gold "{gold_path}" --pred "{pred_path}" '
    f'--db   "{DB_DIR}"   --table "{tables_path}" '
    f'--etype all'
)
print("─  model_1_zeroshot_llama.ipynb:361  model_1_zeroshot_llama27b.ipynb:359  model_2_zeroshot_mistral7b.ipynb:363  model_2_fewshot_mistral.ipynb:395  model_3_fewshot_mistral.ipynb:391  model_4_fewshot_mistral_k1.ipynb:389  model_5_fewshot_mistral_k2.ipynb:389  model_fewshot_llama2chat7b_k3.ipynb:401  model_fewshot_llama2chat7b_k1.ipynb:401 - model_few-shot_llama-2-chat-7b_k5.ipynb:401" * 62)

# ── 15. TỔNG HỢP & IN KẾT QUẢ ───────────────────────────────
all_n  = sum(v["n"]  for v in stats.values())
all_em = sum(v["em"] for v in stats.values())
all_ex = sum(v["ex"] for v in stats.values())

model_size_mb = os.path.getsize(MODEL_PATH) / (1024 ** 2)
gpu_latency_ms = sum(latencies) / len(latencies) * 1000
throughput_sps = len(latencies) / sum(latencies)

sep = "=" * 62
print(f"\n{sep}  model_1_zeroshot_llama.ipynb:373  model_1_zeroshot_llama27b.ipynb:371  model_2_zeroshot_mistral7b.ipynb:375  model_2_fewshot_mistral.ipynb:407  model_3_fewshot_mistral.ipynb:403  model_4_fewshot_mistral_k1.ipynb:401  model_5_fewshot_mistral_k2.ipynb:401  model_fewshot_llama2chat7b_k3.ipynb:413  model_fewshot_llama2chat7b_k1.ipynb:413 - model_few-shot_llama-2-chat-7b_k5.ipynb:413")
print(f"📊  ZEROSHOT TEXTtoSQL  |  Llama27BChat Q4_0  model_1_zeroshot_llama.ipynb:374  model_1_zeroshot_llama27b.ipynb:372  model_2_zeroshot_mistral7b.ipynb:376  model_2_fewshot_mistral.ipynb:408  model_3_fewshot_mistral.ipynb:404  model_4_fewshot_mistral_k1.ipynb:402  model_5_fewshot_mistral_k2.ipynb:402  model_fewshot_llama2chat7b_k3.ipynb:414  model_fewshot_llama2chat7b_k1.ipynb:414 - model_few-shot_llama-2-chat-7b_k5.ipynb:414")
print(f"Prompt : Code Representation (CR_P) + FK + Rule  model_1_zeroshot_llama.ipynb:375  model_1_zeroshot_llama27b.ipynb:373  model_2_zeroshot_mistral7b.ipynb:377  model_2_fewshot_mistral.ipynb:409  model_3_fewshot_mistral.ipynb:405  model_4_fewshot_mistral_k1.ipynb:403  model_5_fewshot_mistral_k2.ipynb:403  model_fewshot_llama2chat7b_k3.ipynb:415  model_fewshot_llama2chat7b_k1.ipynb:415 - model_few-shot_llama-2-chat-7b_k5.ipynb:415")
print(sep)
print(f"{'Difficulty':<12} {'Count':>6}  {'EM (%)':>8}  {'EX (%)':>8}  model_1_zeroshot_llama.ipynb:377  model_1_zeroshot_llama27b.ipynb:375  model_2_zeroshot_mistral7b.ipynb:379  model_2_fewshot_mistral.ipynb:411  model_3_fewshot_mistral.ipynb:407  model_4_fewshot_mistral_k1.ipynb:405  model_5_fewshot_mistral_k2.ipynb:405  model_fewshot_llama2chat7b_k3.ipynb:417  model_fewshot_llama2chat7b_k1.ipynb:417 - model_few-shot_llama-2-chat-7b_k5.ipynb:417")
print(f"{'─'*12}  {'─'*6}  {'─'*8}  {'─'*8}  model_1_zeroshot_llama.ipynb:378  model_1_zeroshot_llama27b.ipynb:376  model_2_zeroshot_mistral7b.ipynb:380  model_2_fewshot_mistral.ipynb:412  model_3_fewshot_mistral.ipynb:408  model_4_fewshot_mistral_k1.ipynb:406  model_5_fewshot_mistral_k2.ipynb:406  model_fewshot_llama2chat7b_k3.ipynb:418  model_fewshot_llama2chat7b_k1.ipynb:418 - model_few-shot_llama-2-chat-7b_k5.ipynb:418")

for d in ["easy", "medium", "hard", "extra", "unknown"]:
    s = stats[d]
    if s["n"] == 0:
        continue
    em_pct = s["em"] / s["n"] * 100
    ex_pct = s["ex"] / s["n"] * 100
    print(f"{d.capitalize():<12} {s['n']:>6}  {em_pct:>7.2f}%  {ex_pct:>7.2f}%  model_1_zeroshot_llama.ipynb:386  model_1_zeroshot_llama27b.ipynb:384  model_2_zeroshot_mistral7b.ipynb:388  model_2_fewshot_mistral.ipynb:420  model_3_fewshot_mistral.ipynb:416  model_4_fewshot_mistral_k1.ipynb:414  model_5_fewshot_mistral_k2.ipynb:414  model_fewshot_llama2chat7b_k3.ipynb:426  model_fewshot_llama2chat7b_k1.ipynb:426 - model_few-shot_llama-2-chat-7b_k5.ipynb:426")

print(f"{'─'*12}  {'─'*6}  {'─'*8}  {'─'*8}  model_1_zeroshot_llama.ipynb:388  model_1_zeroshot_llama27b.ipynb:386  model_2_zeroshot_mistral7b.ipynb:390  model_2_fewshot_mistral.ipynb:422  model_3_fewshot_mistral.ipynb:418  model_4_fewshot_mistral_k1.ipynb:416  model_5_fewshot_mistral_k2.ipynb:416  model_fewshot_llama2chat7b_k3.ipynb:428  model_fewshot_llama2chat7b_k1.ipynb:428 - model_few-shot_llama-2-chat-7b_k5.ipynb:428")
em_all_pct = all_em / all_n * 100 if all_n else 0
ex_all_pct = all_ex / all_n * 100 if all_n else 0
print(f"{'ALL':<12} {all_n:>6}  {em_all_pct:>7.2f}%  {ex_all_pct:>7.2f}%  ←  model_1_zeroshot_llama.ipynb:391  model_1_zeroshot_llama27b.ipynb:389  model_2_zeroshot_mistral7b.ipynb:393  model_2_fewshot_mistral.ipynb:425  model_3_fewshot_mistral.ipynb:421  model_4_fewshot_mistral_k1.ipynb:419  model_5_fewshot_mistral_k2.ipynb:419  model_fewshot_llama2chat7b_k3.ipynb:431  model_fewshot_llama2chat7b_k1.ipynb:431 - model_few-shot_llama-2-chat-7b_k5.ipynb:431")
print(sep)
print(f"{'Model Size':<30} {model_size_mb:>10.2f} MB  model_1_zeroshot_llama.ipynb:393  model_1_zeroshot_llama27b.ipynb:391  model_2_zeroshot_mistral7b.ipynb:395  model_2_fewshot_mistral.ipynb:427  model_3_fewshot_mistral.ipynb:423  model_4_fewshot_mistral_k1.ipynb:421  model_5_fewshot_mistral_k2.ipynb:421  model_fewshot_llama2chat7b_k3.ipynb:433  model_fewshot_llama2chat7b_k1.ipynb:433 - model_few-shot_llama-2-chat-7b_k5.ipynb:433")
print(f"{'Empty SQL':<30} {empty_count:>10} / {all_n}  model_1_zeroshot_llama.ipynb:394  model_1_zeroshot_llama27b.ipynb:392  model_2_zeroshot_mistral7b.ipynb:396  model_2_fewshot_mistral.ipynb:428  model_3_fewshot_mistral.ipynb:424  model_4_fewshot_mistral_k1.ipynb:422  model_5_fewshot_mistral_k2.ipynb:422  model_fewshot_llama2chat7b_k3.ipynb:434  model_fewshot_llama2chat7b_k1.ipynb:434 - model_few-shot_llama-2-chat-7b_k5.ipynb:434")
print(f"{'Inference time':<30} {infer_time/60:>10.1f} min  model_1_zeroshot_llama.ipynb:395  model_1_zeroshot_llama27b.ipynb:393  model_2_zeroshot_mistral7b.ipynb:397  model_2_fewshot_mistral.ipynb:429  model_3_fewshot_mistral.ipynb:425  model_4_fewshot_mistral_k1.ipynb:423  model_5_fewshot_mistral_k2.ipynb:423  model_fewshot_llama2chat7b_k3.ipynb:435  model_fewshot_llama2chat7b_k1.ipynb:435 - model_few-shot_llama-2-chat-7b_k5.ipynb:435")
print(f"{'GPU Latency (ms/sample)':<30} {gpu_latency_ms:>10.2f} ms  model_1_zeroshot_llama.ipynb:396  model_1_zeroshot_llama27b.ipynb:394  model_2_zeroshot_mistral7b.ipynb:398  model_2_fewshot_mistral.ipynb:430  model_3_fewshot_mistral.ipynb:426  model_4_fewshot_mistral_k1.ipynb:424  model_5_fewshot_mistral_k2.ipynb:424  model_fewshot_llama2chat7b_k3.ipynb:436  model_fewshot_llama2chat7b_k1.ipynb:436 - model_few-shot_llama-2-chat-7b_k5.ipynb:436")
print(f"{'Throughput (samples/s)':<30} {throughput_sps:>10.2f} s/s  model_1_zeroshot_llama.ipynb:397  model_1_zeroshot_llama27b.ipynb:395  model_2_zeroshot_mistral7b.ipynb:399  model_2_fewshot_mistral.ipynb:431  model_3_fewshot_mistral.ipynb:427  model_4_fewshot_mistral_k1.ipynb:425  model_5_fewshot_mistral_k2.ipynb:425  model_fewshot_llama2chat7b_k3.ipynb:437  model_fewshot_llama2chat7b_k1.ipynb:437 - model_few-shot_llama-2-chat-7b_k5.ipynb:437")
print(f"{'VRAM Peak':<30} {vram_peak_mb:>10.2f} MB  model_1_zeroshot_llama.ipynb:398  model_1_zeroshot_llama27b.ipynb:396  model_2_zeroshot_mistral7b.ipynb:400  model_2_fewshot_mistral.ipynb:432  model_3_fewshot_mistral.ipynb:428  model_4_fewshot_mistral_k1.ipynb:426  model_5_fewshot_mistral_k2.ipynb:426  model_fewshot_llama2chat7b_k3.ipynb:438  model_fewshot_llama2chat7b_k1.ipynb:438 - model_few-shot_llama-2-chat-7b_k5.ipynb:438")
print(f"{'VRAM Average':<30} {vram_avg_mb:>10.2f} MB  model_1_zeroshot_llama.ipynb:399  model_1_zeroshot_llama27b.ipynb:397  model_2_zeroshot_mistral7b.ipynb:401  model_2_fewshot_mistral.ipynb:433  model_3_fewshot_mistral.ipynb:429  model_4_fewshot_mistral_k1.ipynb:427  model_5_fewshot_mistral_k2.ipynb:427  model_fewshot_llama2chat7b_k3.ipynb:439  model_fewshot_llama2chat7b_k1.ipynb:439 - model_few-shot_llama-2-chat-7b_k5.ipynb:439")
print(sep)

# ── 16. SAVE RESULTS ─────────────────────────────────────────
results = {
    "config": CFG,
    "prompt": "CR_P + FK + PK + no_explanation_rule",
    "model":  "Llama-2-7B-Chat Q4_0",
    "metrics": {
        d: {
            "n":  stats[d]["n"],
            "em": round(stats[d]["em"] / max(stats[d]["n"], 1) * 100, 2),
            "ex": round(stats[d]["ex"] / max(stats[d]["n"], 1) * 100, 2),
        }
        for d in ["easy","medium","hard","extra","unknown"]
        if stats[d]["n"] > 0
    },
    "metrics_all": {
        "n": all_n,
        "em": round(em_all_pct, 2),
        "ex": round(ex_all_pct, 2),
    },
    "perf": {
        "model_size_mb":   round(model_size_mb, 2),
        "empty_sql":       empty_count,
        "infer_time_min":  round(infer_time / 60, 2),
        "latency_ms":      round(gpu_latency_ms, 2),
        "throughput_sps":  round(throughput_sps, 2),
        "vram_peak_mb":    round(vram_peak_mb, 2),
        "vram_avg_mb":     round(vram_avg_mb, 2),
    },
}
with open(os.path.join(RESULT_DIR, "results_v5.json"),
          'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"\n✅ Results saved → {RESULT_DIR}  model_1_zeroshot_llama.ipynb:435  model_1_zeroshot_llama27b.ipynb:433  model_2_zeroshot_mistral7b.ipynb:437  model_2_fewshot_mistral.ipynb:469  model_3_fewshot_mistral.ipynb:465  model_4_fewshot_mistral_k1.ipynb:463  model_5_fewshot_mistral_k2.ipynb:463  model_fewshot_llama2chat7b_k3.ipynb:475  model_fewshot_llama2chat7b_k1.ipynb:475 - model_few-shot_llama-2-chat-7b_k5.ipynb:475")